In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ==========================================
# 1. LOAD DATA & PREPARE 3D SEQUENCES
# ==========================================
df_dl_v3 = pd.read_csv("../data/processed/demand_features_v3.csv", parse_dates=["date"])

feature_cols_v3 = ['year', 'month', 'day_of_week', 'is_weekend', 'is_back_to_school', 'is_holiday_season', 
                   'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_mean_30', 'is_promo', 'is_stockout']

SEQ_LENGTH = 14 # Look back 14 days to predict the next day

all_X_train, all_y_train = [], []
all_X_test, all_y_test, all_actual_test, all_lag_test = [], [], [], []

for sku in df_dl_v3['sku_id'].unique():
    sku_df = df_dl_v3[df_dl_v3['sku_id'] == sku].reset_index(drop=True)
    
    # Neural Networks require scaled features to prevent exploding gradients
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(sku_df[feature_cols_v3])
    
    target = sku_df['units_sold_diff'].values
    actual = sku_df['units_sold'].values
    lag_1 = sku_df['lag_1'].values
    
    X, y, actuals, lags = [], [], [], []
    
    # Create sliding windows of 14 days
    for i in range(len(sku_df) - SEQ_LENGTH):
        X.append(scaled_features[i : i + SEQ_LENGTH])
        y.append(target[i + SEQ_LENGTH])
        actuals.append(actual[i + SEQ_LENGTH])
        lags.append(lag_1[i + SEQ_LENGTH])
        
    X, y, actuals, lags = np.array(X), np.array(y), np.array(actuals), np.array(lags)
    
    # 80/20 chronological split per SKU
    split_idx = int(len(X) * 0.8)
    
    all_X_train.append(X[:split_idx])
    all_y_train.append(y[:split_idx])
    all_X_test.append(X[split_idx:])
    all_y_test.append(y[split_idx:])
    all_actual_test.append(actuals[split_idx:])
    all_lag_test.append(lags[split_idx:])

# Convert to PyTorch Tensors
X_train = torch.tensor(np.concatenate(all_X_train), dtype=torch.float32)
y_train = torch.tensor(np.concatenate(all_y_train), dtype=torch.float32)
X_test = torch.tensor(np.concatenate(all_X_test), dtype=torch.float32)

actual_test = np.concatenate(all_actual_test)
lag_test = np.concatenate(all_lag_test)

# Create DataLoader for batching
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)


# ==========================================
# 2. DEFINE THE TRANSFORMER ARCHITECTURE
# ==========================================
class TimeSeriesTransformer(nn.Module):
    def __init__(self, num_features, d_model=64, nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        # 1. Project input tabular features to a higher-dimensional embedding (d_model)
        self.input_projection = nn.Linear(num_features, d_model)
        
        # 2. Transformer Encoder Layer (Learns relationships across the 14-day sequence)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dropout=dropout, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. Final output layer predicting the differenced target value
        self.decoder = nn.Linear(d_model, 1)
        
    def forward(self, x):
        # x shape: (batch_size, seq_length, num_features)
        x = self.input_projection(x)
        x = self.transformer(x)
        
        # We only care about forecasting the next day, so we take the output at the last time step
        last_step_output = x[:, -1, :] 
        out = self.decoder(last_step_output)
        return out.squeeze()


# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
model_tft = TimeSeriesTransformer(num_features=len(feature_cols_v3))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_tft.parameters(), lr=0.001)

epochs = 20
print("Starting Training...")
model_tft.train()

for epoch in range(epochs):
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model_tft(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")


# ==========================================
# 4. EVALUATE PREDICTIONS
# ==========================================
model_tft.eval()
with torch.no_grad():
    # Model predicts the difference
    test_preds_diff = model_tft(X_test).numpy()
    
# Reconstruct actual values
preds_actual_tft = lag_test + test_preds_diff

# Calculate Metrics
rmse_tft = np.sqrt(mean_squared_error(actual_test, preds_actual_tft))
mae_tft = mean_absolute_error(actual_test, preds_actual_tft)
mape_tft = np.mean(np.abs((actual_test - preds_actual_tft) / actual_test)) * 100
r2_tft = r2_score(actual_test, preds_actual_tft)

print("\n--- PyTorch Transformer Performance ---")
print(f"RMSE: {rmse_tft:.3f}")
print(f"MAE:  {mae_tft:.3f}")
print(f"MAPE: {mape_tft:.3f}%")
print(f"R2:   {r2_tft:.3f}")

Starting Training...
Epoch 5/20 | Loss: 5942.5465
Epoch 10/20 | Loss: 5346.6078
Epoch 15/20 | Loss: 4854.8126
Epoch 20/20 | Loss: 4397.9075

--- PyTorch Transformer Performance ---
RMSE: 93.036
MAE:  48.726
MAPE: 19.743%
R2:   0.429


In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings("ignore")

# 1. HARDWARE ACCELERATION
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device.type.upper()}")

print("Loading 1-Million-Row V4 Dataset...")
df = pd.read_csv("../data/processed/demand_features_v4_1M.csv", parse_dates=["date"])

# 2. FEATURE ENGINEERING & ENCODING
le = LabelEncoder()
df['store_id_encoded'] = le.fit_transform(df['store_id'])

feature_cols = [
    'store_id_encoded', 'year', 'month', 'day_of_week', 'is_weekend', 
    'is_holiday_season', 'lag_1', 'lag_7', 'rolling_mean_7', 
    'rolling_mean_30', 'is_promo', 'is_stockout'
]

all_results_pytorch = []

# 3. DEFINE THE NEURAL NETWORK ARCHITECTURE
class DemandNet(nn.Module):
    def __init__(self, input_size):
        super(DemandNet, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            nn.Linear(32, 1) # Output layer: predicting units_sold_diff
        )

    def forward(self, x):
        return self.network(x)

# 4. TRAINING PIPELINE PER SKU
for sku in df["sku_id"].unique():
    print(f"\n--- Training PyTorch DNN for {sku} ---")
    sku_df = df[df["sku_id"] == sku].sort_values('date').reset_index(drop=True)
    
    # 80/20 Chronological Split
    split_idx = int(len(sku_df) * 0.8)
    train_df = sku_df.iloc[:split_idx].reset_index(drop=True)
    test_df = sku_df.iloc[split_idx:].reset_index(drop=True)
    
    X_train, y_train = train_df[feature_cols].values, train_df["units_sold_diff"].values
    X_test, y_test = test_df[feature_cols].values, test_df["units_sold_diff"].values
    
    # Neural Networks REQUIRE feature scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Convert to PyTorch Tensors
    X_train_tensor = torch.FloatTensor(X_train_scaled).to(device)
    y_train_tensor = torch.FloatTensor(y_train).view(-1, 1).to(device)
    X_test_tensor = torch.FloatTensor(X_test_scaled).to(device)
    
    # Create DataLoaders for batching
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=4096, shuffle=True)
    
    # Initialize Model, Loss Function, and Optimizer
    model = DemandNet(input_size=len(feature_cols)).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-5)
    
    # 5. TRAINING LOOP
    epochs = 20
    model.train()
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")
            
    # 6. EVALUATION
    model.eval()
    with torch.no_grad():
        test_preds_diff = model(X_test_tensor).cpu().numpy().flatten()
        
    # Reconstruct actual sales and apply a floor of 0
    test_lag_1 = test_df["lag_1"].values
    y_test_actual = test_df["units_sold"].values
    final_preds_actual = np.maximum(0, test_lag_1 + test_preds_diff)
    
    # Calculate Metrics
    rmse = np.sqrt(mean_squared_error(y_test_actual, final_preds_actual))
    mae = mean_absolute_error(y_test_actual, final_preds_actual)
    mape = np.mean(np.abs((y_test_actual - final_preds_actual) / y_test_actual)) * 100
    r2 = r2_score(y_test_actual, final_preds_actual)
    
    all_results_pytorch.append({
        "model": "PyTorch_DNN_1M",
        "sku_id": sku,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    })

# 7. PRINT FINAL REPORT
print("\n=== FINAL PYTORCH PERFORMANCE ===")
summary_df = pd.DataFrame(all_results_pytorch).groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)
print(summary_df)

Using compute device: CPU
Loading 1-Million-Row V4 Dataset...

--- Training PyTorch DNN for MBA13 ---
Epoch 5/20 | Loss: 112.9979
Epoch 10/20 | Loss: 112.0999
Epoch 15/20 | Loss: 112.5361
Epoch 20/20 | Loss: 111.2195

--- Training PyTorch DNN for MBA15 ---
Epoch 5/20 | Loss: 113.3318
Epoch 10/20 | Loss: 111.9826
Epoch 15/20 | Loss: 112.0713
Epoch 20/20 | Loss: 111.3276

--- Training PyTorch DNN for MBP14 ---
Epoch 5/20 | Loss: 112.9364
Epoch 10/20 | Loss: 111.8765
Epoch 15/20 | Loss: 111.6805
Epoch 20/20 | Loss: 111.8596

--- Training PyTorch DNN for MBP16 ---
Epoch 5/20 | Loss: 113.1694
Epoch 10/20 | Loss: 112.0180
Epoch 15/20 | Loss: 111.7478
Epoch 20/20 | Loss: 110.9397

=== FINAL PYTORCH PERFORMANCE ===
                 RMSE    MAE    MAPE     R2
model                                      
PyTorch_DNN_1M  10.37  8.133  17.696  0.454
